In [ ]:
# GPU'yu kontrol et
!nvidia-smi

# Unsloth ve gerekli kütüphaneleri kur
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps xformers trl peft accelerate bitsandbytes

Fri Apr  3 13:34:49 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   32C    P0             46W /  400W |       0MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [ ]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048
dtype = None # Otomatik algılama
load_in_4bit = True # VRAM tasarrufu için

# Llama 3.1 8B modelini yüklüyoruz
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Meta-Llama-3.1-8B-bnb-4bit", # 🚀 Sadece bu satır güncellendi
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

# LoRA Adaptörlerini ekliyoruz
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
    use_rslora = False,
    loftq_config = None,
)
print("✅ Llama 3.1 ve LoRA adaptörleri başarıyla yüklendi!")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
Unsloth: Your Flash Attention 2 installation seems to be broken. Using Xformers instead. No performance changes will be seen.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.4.1: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    NVIDIA A100-SXM4-40GB. Num GPUs = 1. Max memory: 39.494 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/5.70G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/235 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/459 [00:00<?, ?B/s]

Unsloth: Will load unsloth/Meta-Llama-3.1-8B-bnb-4bit as a legacy tokenizer.
Unsloth 2026.4.1 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.


✅ Llama 3.1 ve LoRA adaptörleri başarıyla yüklendi!


In [ ]:
# 2. Drive Bağlantısı ve Model Yükleme
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# ==========================================
# 4. Veri Seti Yükleme ve Formatlama (GÜNCELLENDİ)
# ==========================================
from datasets import load_dataset

alpaca_prompt = """Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
{}

### Input:
{}

### Response:
{}"""

EOS_TOKEN = tokenizer.eos_token

def formatting_prompts_func(examples):
    instructions = examples["instruction"]
    inputs       = examples["input"]
    outputs      = examples["output"]
    texts = []
    for instruction, input, output in zip(instructions, inputs, outputs):
        text = alpaca_prompt.format(instruction, input, output) + EOS_TOKEN
        texts.append(text)
    return { "text" : texts, }

# 🚀 DÜZELTME: Hem Train hem de Validation dosyalarını aynı anda okuyoruz
dataset = load_dataset("json", data_files={
    "train": "mizan_v4_train.jsonl",
    "validation": "mizan_v4_validation.jsonl"
})

# İki seti de Alpaca formatına dönüştürüyoruz
train_dataset = dataset["train"].map(formatting_prompts_func, batched = True)
eval_dataset = dataset["validation"].map(formatting_prompts_func, batched = True)

print(f"📦 Eğitim (Train) Verisi: {len(train_dataset)} satır")
print(f"🧐 Doğrulama (Validation) Verisi: {len(eval_dataset)} satır")


# ==========================================
# 5. Eğitim (Fine-Tuning) (GÜNCELLENDİ)
# ==========================================
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = train_dataset, # Formatlanmış Train seti
    eval_dataset = eval_dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        num_train_epochs = 1, # Tüm veri setini 1 kez tam dolaşır
        learning_rate = 2e-4,
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 10, # Ekrana her 10 adımda bir log basar
        eval_strategy = "steps",
        eval_steps = 100, # 🚀 DÜZELTME: Her 100 eğitim adımında bir Validation setini test et
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
    ),
)

print("🚀 EĞİTİM BAŞLIYOR...")
trainer_stats = trainer.train()

Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/11207 [00:00<?, ? examples/s]

Map:   0%|          | 0/1400 [00:00<?, ? examples/s]

📦 Eğitim (Train) Verisi: 11207 satır
🧐 Doğrulama (Validation) Verisi: 1400 satır


...

Unsloth: Tokenizing ["text"] (num_proc=16):   0%|          | 0/1400 [00:00<?, ? examples/s]

🚀 EĞİTİM BAŞLIYOR...


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 11,207 | Num Epochs = 1 | Total steps = 1,401
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 41,943,040 of 8,072,204,288 (0.52% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Step,Training Loss,Validation Loss
100,0.228397,0.249005
200,0.206541,0.237156
300,0.232489,0.233048
400,0.244125,0.230596
500,0.230197,0.228715
600,0.252037,0.226955
700,0.262097,0.225651
800,0.218249,0.224087
900,0.196798,0.222294
1000,0.249328,0.221019


/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:172: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API 

In [ ]:
# ==========================================
# 6. Model Kayıt İşlemleri (LoRA ve GGUF)
# ==========================================

# 1. Önce sadece LoRA ağırlıklarını Drive'a kaydedelim
lora_path = "/content/drive/MyDrive/Mizan_V4_Model/LoRA"
model.save_pretrained(lora_path)
tokenizer.save_pretrained(lora_path)
print(f"💾 LoRA Ağırlıkları Başarıyla Kaydedildi:\n👉 {lora_path}")

print("-" * 50)

# 2. Şimdi modeli GGUF (Q4_K_M Kuantizasyonu) formatına çevirip Drive'a kaydedelim
gguf_path = "/content/drive/MyDrive/Mizan_V4_Model/GGUF"
print("⏳ GGUF formatına dönüştürülüyor ve Drive'a kaydediliyor...")
print("⚠️ Lütfen bekleyin, bu işlem Colab T4 GPU'da 10-15 dakika kadar sürebilir!")

model.save_pretrained_gguf(
    gguf_path,
    tokenizer,
    quantization_method = "q4_k_m" # Hem hızlı hem mantıklı bir sıkıştırma formatı (Ollama için ideal)
)

print(f"🎉 MUHTEŞEM! GGUF Modeli Başarıyla Kaydedildi:\n👉 {gguf_path}")
print("🚀 Takım 111 için Mizan V4 artık dünyayı düzeltmeye hazır!")

💾 LoRA Ağırlıkları Başarıyla Kaydedildi:
👉 /content/drive/MyDrive/Mizan_V4_Model/LoRA
--------------------------------------------------
⏳ GGUF formatına dönüştürülüyor ve Drive'a kaydediliyor...
⚠️ Lütfen bekleyin, bu işlem Colab T4 GPU'da 10-15 dakika kadar sürebilir!
Unsloth: Merging model weights to 16-bit format...


config.json:   0%|          | 0.00/947 [00:00<?, ?B/s]

RuntimeError: Failed to save/merge model: Unsloth: Failed saving locally - no disk space left. Uploading can work luckily! Use .push_to_hub instead.